# انسانی ان پٹ: پری-ایکشن گیٹس، رسک ٹیرنگ، اور آڈٹ لاگنگ

اس سبق کے README میں انسانی ان پٹ کو ایک مختصر مثال کے ساتھ پیش کیا گیا ہے جو صارف سے ایجنٹ کے ردعمل کے بعد `APPROVE` یا `REJECT` مانگتی ہے۔ یہ طریقہ کار شروع کرنے کے لیے اچھا ہے، لیکن عام پروڈکشن HITL تنصیبات کو عام طور پر تین اضافی چیزوں کی ضرورت ہوتی ہے:

1. ایک **پری-ایکشن گیٹ** جو **ایجنٹ کے کسی خطرناک قدم اٹھانے سے پہلے** چلتی ہے، تاکہ لاگت، ناقابل واپسی اور تاخیر قابو میں رہیں۔
2. **رسک ٹیرنگ**، تاکہ کم خطرناک اقدامات خود بخود انجام پائیں، درمیانے خطرے کے اقدامات بیچ میں منظور کیے جائیں، اور صرف زیادہ خطرناک اقدامات انسانی تصدیق کا تقاضا کریں۔
3. ایک **آڈٹ لاگ اور نظرِ ثانی کا دائرہ**، تاکہ ہر گیٹ کا فیصلہ JSONL میں ریکارڈ کیا جائے، اور رد کرنے پر ایجنٹ کو ساختی وجہ کے ساتھ دوبارہ دعوت دی جائے نہ کہ صرف `Revising...` پرنٹ کیا جائے۔

یہ نوٹ بک `06-system-message-framework.ipynb` کے وہی بنیادی اجزاء استعمال کرتے ہوئے ان تمام کو ترتیب وار تیار کرتی ہے۔ یہ `DEMO_MODE = True` (کوئی متعامل ان پٹ نہ چاہیے) یا حقیقی `input()` پر مبنی ان پٹ کے ساتھ `DEMO_MODE = False` میں مکمل چلائی جا سکتی ہے۔ نوٹ: DEMO_MODE میں تیسرے مقصد کی دوبارہ کوشش اسکرپٹ کی گئی ہے تاکہ دائرے کے عمل دکھائی دیں۔ حقیقی نظرثانی پر مبنی دوبارہ درجہ بندی کے لیے `DEMO_MODE = False` اور ایک آپریٹر کی ضرورت ہوتی ہے۔

**دائرہ کار سے باہر (دیگر اسباق میں ہینڈل کیا گیا):** تصدیق اور رسائی کنٹرول (سبق 06 README خطرہ 2)، ٹول-کال مڈل ویئر (سبق 14 MAF گہری وضاحت)، ملٹی-ایجنٹ مباحثے کے پیٹرن۔


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## پیٹرن 1: پُر عمل دروازہ

README کے HITL کا ٹکڑا پہلے ایجنٹ کو کال کرتا ہے، پھر صارف سے آؤٹ پٹ کی منظوری مانگتا ہے۔ یہ ایک **بعد از عمل** بہاؤ ہے۔ ایجنٹ پہلے ہی عمل کر چکا ہے، اس لیے LLM کال کی قیمت پہلے ہی ادا ہو چکی ہے، اور کوئی بھی ضمنی اثر (ای میل بھیجی گئی، ڈیٹا بیس کی قطار لکھی گئی، تبصرہ پوسٹ کیا گیا) پہلے ہی ہو چکا ہے۔

ایک **قبل از عمل** بہاؤ اس دروازے کو اس سے پہلے داخل کرتا ہے کہ ایجنٹ خطرناک مرحلہ چلائے۔ ایجنٹ کارروائی کا مشورہ دیتا ہے، دروازہ فیصلہ کرتا ہے کہ آیا اسے عملدرآمد کرنا ہے، اور صرف منظوری پر ضمنی اثر ہوتا ہے۔

| پہلو | بعد از عمل منظوری (README ٹکڑا) | قبل از عمل دروازہ (اس نوٹ بک) |
|---|---|---|
| منظوری کب چلتی ہے؟ | ایجنٹ کے آؤٹ پٹ تیار کرنے کے بعد | کسی بھی ضمنی اثر کے عمل سے پہلے |
| مستردگی پر LLM کی لاگت | پہلے ہی ادا کی جا چکی ہے | صرف مشورے کے لیے ادا کی جاتی ہے، کارروائی کے لیے نہیں |
| ناقابل واپسی ضمنی اثرات | ممکن ہے (کارروائی پہلے ہی ہو چکی ہے) | روکی گئی |
| آڈٹ کی وضاحت | منظوری ایک پرنٹ بیان ہے | منظوری ایک JSON ریکارڈ ہے جس میں ٹائم اسٹیمپ، کارروائی، وجہ شامل ہے |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## نمونہ 3: آڈٹ لوگ اور نظرثانی کا لوپ

ایک `print("Response approved.")` آڈٹ لوگ نہیں ہے۔ بھروسے کے لیے، ہر گیٹ کے فیصلے کو ایک ساختہ کردہ واقعے کے طور پر ریکارڈ کیا جانا چاہیے جسے آپ بعد میں تلاش کر سکیں، دوبارہ چلائیں، یا واقعہ کے جائزے سے منسلک کریں۔

دو حصے:

1. **صرف اضافہ کرنے والا JSONL۔** ہر فیصلے کے لیے ایک لائن، جس میں ٹائم اسٹیمپ، عمل، طبقہ، فیصلہ، وجہ شامل ہو۔ اسے تلاش کرنا آسان ہے، اور بعد میں کسی حقیقی لوگ اسٹور میں بھیجنا آسان ہے۔
2. **رد کرنے پر نظرثانی کا لوپ۔** جب گیٹ `deny` واپس کرتا ہے، ایجنٹ خود کو رد کرنے کی وجہ کے ساتھ دوبارہ پرامپٹ کرتا ہے تاکہ اگلی تجویز مسئلہ سے بچ سکے۔


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## اضافی وسائل

کئی اور عوامی پروجیکٹس ان HITL نمونوں کے مختلف ورژنز لاگو کرتے ہیں۔ اپنے اسٹیک کے مطابق فٹ بیٹھنے والے طریقوں کا موازنہ کریں:

- **LangChain** human-in-the-loop ٹول ریپرز ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ایک ایسے ٹول ریپرز جو انسانی ان پٹ کے لیے عمل کو روک دیتے ہیں۔
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ نے اس کی ساخت بدلی): ملٹی ایجنٹ بات چیت میں انسانی نمائندگی کے لیے ایک ایجنٹ رول استعمال کرتا ہے۔
- **Microsoft Agent Framework (MAF)** فنکشن-انوکیشن مڈل ویئر ([docs](https://learn.microsoft.com/agent-framework/)): ہر ٹول/فنکشن کال کے ارد گرد چلنے والا مڈل ویئر، گیٹنگ لاجک اور منظوری کے بہاؤ کے لیے موزوں۔

ہر پروجیکٹ تین ذیلی نمونوں کو مختلف انداز میں ہینڈل کرتا ہے: LangChain انہیں ٹولز کے طور پر لپیٹتا ہے، AutoGen ایجنٹ رول استعمال کرتا ہے، اور Microsoft Agent Framework فنکشن-انوکیشن مڈل ویئر استعمال کرتا ہے۔ اپنی ایجنٹ کے لیے ڈیزائن منتخب کرنے سے پہلے ایک یا دو عملدرآمد کو مکمل طور پر پڑھیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
